In [3]:
# ============================================================
# INSTALL APACHE KAFKA IN GOOGLE COLAB
# ============================================================

import os
import subprocess
import time

KAFKA_VERSION = "3.9.1"
SCALA_VERSION = "2.13"

KAFKA_FOLDER = f"kafka_{SCALA_VERSION}-{KAFKA_VERSION}"
KAFKA_TGZ = f"{KAFKA_FOLDER}.tgz"

# Download Kafka
!wget -q --show-progress \
    https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/{KAFKA_TGZ}

# Extract Kafka
!tar -xzf {KAFKA_TGZ}

# Check that Kafka was extracted
print("\nKafka directory:")
!ls -la {KAFKA_FOLDER}/bin | head -20

KAFKA_DIR = f"/content/{KAFKA_FOLDER}"

print("\nKafka path:", KAFKA_DIR)

# Verify kafka-storage.sh exists
storage_script = f"{KAFKA_DIR}/bin/kafka-storage.sh"

if os.path.exists(storage_script):
    print(" kafka-storage.sh found")
else:
    print(" kafka-storage.sh NOT found")


kafka_2.13-3.9.1.tg 100%[===================>] 116.45M   300KB/s    in 6m 33s  

Kafka directory:
total 188
drwxr-xr-x 3 root root  4096 May 12  2025 .
drwxr-xr-x 7 root root  4096 May 12  2025 ..
-rwxr-xr-x 1 root root  1423 May 12  2025 connect-distributed.sh
-rwxr-xr-x 1 root root  1396 May 12  2025 connect-mirror-maker.sh
-rwxr-xr-x 1 root root   963 May 12  2025 connect-plugin-path.sh
-rwxr-xr-x 1 root root  1464 May 12  2025 connect-standalone.sh
-rwxr-xr-x 1 root root   861 May 12  2025 kafka-acls.sh
-rwxr-xr-x 1 root root   873 May 12  2025 kafka-broker-api-versions.sh
-rwxr-xr-x 1 root root   880 May 12  2025 kafka-client-metrics.sh
-rwxr-xr-x 1 root root   871 May 12  2025 kafka-cluster.sh
-rwxr-xr-x 1 root root   864 May 12  2025 kafka-configs.sh
-rwxr-xr-x 1 root root   965 May 12  2025 kafka-console-consumer.sh
-rwxr-xr-x 1 root root   944 May 12  2025 kafka-console-producer.sh
-rwxr-xr-x 1 root root   897 May 12  2025 kafka-consumer-groups.sh
-rwxr-xr-x 1 root root   959 

In [4]:

# START KAFKA BROKER

import subprocess
import time
import os

# Generate Kafka cluster ID
cluster_id = subprocess.check_output(
    [f"{KAFKA_DIR}/bin/kafka-storage.sh", "random-uuid"],
    text=True
).strip()

print("Kafka Cluster ID:", cluster_id)

# Format Kafka storage
subprocess.run(
    [
        f"{KAFKA_DIR}/bin/kafka-storage.sh",
        "format",
        "-t",
        cluster_id,
        "-c",
        f"{KAFKA_DIR}/config/kraft/server.properties"
    ],
    check=True
)

# Start Kafka
kafka_log = open("/content/kafka.log", "w")

kafka_process = subprocess.Popen(
    [
        f"{KAFKA_DIR}/bin/kafka-server-start.sh",
        f"{KAFKA_DIR}/config/kraft/server.properties"
    ],
    stdout=kafka_log,
    stderr=subprocess.STDOUT
)

# Give Kafka some time to start
time.sleep(10)

print("Kafka broker started!")
print("Kafka server: localhost:9092")


Kafka Cluster ID: [0.028s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.028s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
nvYjJ65vTsuKkOrtk-jJog
Kafka broker started!
Kafka server: localhost:9092


In [5]:
!ps aux | grep kafka


root        5059 27.3  2.7 3757028 371572 ?      Sl   13:14   0:08 java -Xmx1G -Xms1G -server -XX:+UseG1GC -XX:MaxGCPauseMillis=20 -XX:InitiatingHeapOccupancyPercent=35 -XX:+ExplicitGCInvokesConcurrent -XX:MaxInlineLevel=15 -Djava.awt.headless=true -Xlog:gc*:file=/content/kafka_2.13-3.9.1/bin/../logs/kafkaServer-gc.log:time,tags:filecount=10,filesize=100M -Dcom.sun.management.jmxremote=true -Dcom.sun.management.jmxremote.authenticate=false -Dcom.sun.management.jmxremote.ssl=false -Dkafka.logs.dir=/content/kafka_2.13-3.9.1/bin/../logs -Dlog4j.configuration=file:/content/kafka_2.13-3.9.1/bin/../config/log4j.properties -cp /content/kafka_2.13-3.9.1/bin/../libs/activation-1.1.1.jar:/content/kafka_2.13-3.9.1/bin/../libs/aopalliance-repackaged-2.6.1.jar:/content/kafka_2.13-3.9.1/bin/../libs/argparse4j-0.7.0.jar:/content/kafka_2.13-3.9.1/bin/../libs/audience-annotations-0.12.0.jar:/content/kafka_2.13-3.9.1/bin/../libs/caffeine-2.9.3.jar:/content/kafka_2.13-3.9.1/bin/../libs/commons-beanutils-

In [6]:
!pip install kafka-python-ng -q


In [7]:
from kafka import KafkaProducer, KafkaConsumer

print("kafka-python installed")

kafka-python installed


In [8]:
!{KAFKA_DIR}/bin/kafka-topics.sh \
    --create \
    --topic orders \
    --bootstrap-server localhost:9092 \
    --partitions 3 \
    --replication-factor 1


[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Created topic orders.


In [9]:
!{KAFKA_DIR}/bin/kafka-topics.sh \
    --describe \
    --topic orders \
    --bootstrap-server localhost:9092


[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Topic: orders	TopicId: ful559s4RLOqW4vyCf9KFA	PartitionCount: 3	ReplicationFactor: 1	Configs: segment.bytes=1073741824
	Topic: orders	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: orders	Partition: 1	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: orders	Partition: 2	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 


In [11]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    key_serializer=lambda key: str(key).encode("utf-8"),
    value_serializer=lambda value: json.dumps(value).encode("utf-8")
)

print(" Producer created")


 Producer created


In [12]:
records = [
    {"order_id": 101, "product": "Laptop", "price": 75000},
    {"order_id": 102, "product": "Mouse", "price": 1200},
    {"order_id": 103, "product": "Keyboard", "price": 2500},
    {"order_id": 104, "product": "Monitor", "price": 15000},
    {"order_id": 105, "product": "Headphones", "price": 3500},
    {"order_id": 106, "product": "Webcam", "price": 4500},
]

print("========== PRODUCER ==========")

for record in records:

    future = producer.send(
        "orders",
        key=record["order_id"],
        value=record
    )

    metadata = future.get(timeout=10)

    print(
        f"Record: {record} | "
        f"Partition: {metadata.partition} | "
        f"Offset: {metadata.offset}"
    )

producer.flush()


========== PRODUCER ==========
Record: {'order_id': 101, 'product': 'Laptop', 'price': 75000} | Partition: 2 | Offset: 0
Record: {'order_id': 102, 'product': 'Mouse', 'price': 1200} | Partition: 0 | Offset: 0
Record: {'order_id': 103, 'product': 'Keyboard', 'price': 2500} | Partition: 2 | Offset: 1
Record: {'order_id': 104, 'product': 'Monitor', 'price': 15000} | Partition: 2 | Offset: 2
Record: {'order_id': 105, 'product': 'Headphones', 'price': 3500} | Partition: 2 | Offset: 3
Record: {'order_id': 106, 'product': 'Webcam', 'price': 4500} | Partition: 0 | Offset: 1


In [14]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    "orders",
    bootstrap_servers="localhost:9092",
    group_id="order-consumer-group",
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    key_deserializer=lambda key: int(key.decode("utf-8")),
    value_deserializer=lambda value: json.loads(value.decode("utf-8"))
)

print(" Consumer created")


 Consumer created


In [15]:
print("========== CONSUMER ==========")

count = 0

for message in consumer:

    print("\nRecord received")
    print("------------------------")
    print("Topic     :", message.topic)
    print("Partition :", message.partition)
    print("Offset    :", message.offset)
    print("Key       :", message.key)
    print("Value     :", message.value)

    count += 1

    if count == 6:
        break

consumer.close()


========== CONSUMER ==========



Record received
------------------------
Topic     : orders
Partition : 2
Offset    : 0
Key       : 101
Value     : {'order_id': 101, 'product': 'Laptop', 'price': 75000}

Record received
------------------------
Topic     : orders
Partition : 2
Offset    : 1
Key       : 103
Value     : {'order_id': 103, 'product': 'Keyboard', 'price': 2500}

Record received
------------------------
Topic     : orders
Partition : 2
Offset    : 2
Key       : 104
Value     : {'order_id': 104, 'product': 'Monitor', 'price': 15000}

Record received
------------------------
Topic     : orders
Partition : 2
Offset    : 3
Key       : 105
Value     : {'order_id': 105, 'product': 'Headphones', 'price': 3500}

Record received
------------------------
Topic     : orders
Partition : 0
Offset    : 0
Key       : 102
Value     : {'order_id': 102, 'product': 'Mouse', 'price': 1200}

Record received
------------------------
Topic     : orders
Partition : 0
Offset    : 1
Key       : 106
Value     : {'order_id': 106, 'p